# Post-processing avec OPTICS

Data utilisé : pre-processing effectué avec algorithme de Louise (CSV) + drift correction avec ImageJ (cross-correlation)

In [9]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

# Imports 
%matplotlib qt
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import DBSCAN
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from matplotlib.colors import hsv_to_rgb
import tifffile as tiff
import math
from sklearn.cluster import OPTICS
import matplotlib.gridspec as gridspec
from tqdm import tqdm
import matplotlib

In [45]:
# Recuperation of data
data = pd.read_csv(r'C:\Users\LOCCO\Project_Curie\pour_lola\zone1_cc.csv',sep=',')
print(data.columns)

mask = tiff.imread(r"C:\Users\LOCCO\Project_Curie\pour_lola\zone1_mask.tif")    
mask = np.array(mask.transpose()) #Fiji écrit en y,x

Index(['frame', 'x [nm]', 'y [nm]', 'z [nm]', 'intensity', 'bkd', 'resnorm',
       'sigmax [nm]', 'sigmay [nm]', 'sigmaz [nm]', 'delta', 'deltaz1',
       'deltaz3', 'rho', 'rhoz1', 'rhoz3', 'deltaz', 'rhoz'],
      dtype='str')


In [46]:
#Import data
frame = data['frame'].values
X = data[['x [nm]', 'y [nm]', 'z [nm]']].values
rho = data['rho'].values
delta = data['delta'].values
N_photons = data['intensity'].values
sigma = data[['sigmax [nm]', 'sigmay [nm]', 'sigmaz [nm]']].values

#Apply mask 12 = taille pixel, 5 = magnification imageJ
#mask_vect = (mask[(X[:, 0] / (120/5)).astype(int),(X[:, 1] / (120/5)).astype(int)] > 0)&(sigma[:,0] <= 240) & (sigma[:,1] <= 240) & (sigma[:,2] <= 720)
ix = np.clip((X[:, 0] / (120/5)).astype(int), 0, mask.shape[0] - 1)
iy = np.clip((X[:, 1] / (120/5)).astype(int), 0, mask.shape[1] - 1)

mask_vect = (mask[ix, iy] > 0) &(sigma[:,0] <= 240) & (sigma[:,1] <= 240) & (sigma[:,2] <= 720)

X_masked = X[mask_vect]
sigma_masked   = sigma[mask_vect]
rho_masked     = rho[mask_vect]
delta_masked   = delta[mask_vect]
frame_masked   = frame[mask_vect]

print('after masking free-hand roi & noisy outliers data set is ', len(X_masked), ' instead of', len(X))

after masking free-hand roi & noisy outliers data set is  139803  instead of 340506


In [47]:
#Arbitrary threshhold for merging localizations in consecutive frames 

def recursive_call(not_counted,i,X, rho, delta, frame, previous_index,th_lat=50, th_axial=75):
    #i is index of the frame we are in 
    #Let's create a mask that takes into account only the next frame and that looks for points at <th_lat and <th_ax
    mask = (frame==frame[i]+1) & ((X[:,0]-X[i,0])**2+(X[:,1]-X[i,1])**2<th_lat**2) & ((X[:,2]-X[i,2])**2<th_axial**2)
    #Is there at least one True value in mask?
    if (mask==True).any():
        not_counted[i] = False
        return recursive_call(not_counted, np.where(mask)[0][0], X, rho, delta, frame, previous_index=np.concatenate((previous_index, np.where(mask)[0])),th_lat=50, th_axial=75)
    else:
        return previous_index.astype(int)

In [62]:
#Commençons par obtenir les std obtenus avec les threshholds arbitraires voir si ça tient la route
#Init
nb_id = len(X_masked[:,0])
not_counted = np.ones(nb_id, dtype=bool)
stdx = []
stdy = []
stdz = []
stdrho = []
stddelta = []

th_lat= 5
th_axial = 13 #Obtenus après calcul de std


for i in tqdm(range(nb_id), desc="Processing points"):
    if not_counted[i]:
        indices =  recursive_call(not_counted,i,X_masked, rho_masked, delta_masked, frame_masked, previous_index=np.array([i]),th_lat=th_lat, th_axial= th_axial)
        if len(indices)>1:
            #Pour le calcul de threshhold*
            
            stdx.append(np.std([X_masked[j,0] for j in indices]))
            stdy.append(np.std([X_masked[j,1] for j in indices]))
            stdz.append(np.std([X_masked[j,2] for j in indices]))
            stdrho.append(np.std(rho_masked[indices]))
            stddelta.append(np.std(delta_masked[indices]))
            
            '''
            #Collapse les doublons into their average position, keep it at the last index, and erase the others by setting them to NaN.
            X_masked[indices[-1], 0] = np.mean(X_masked[indices, 0])
            X_masked[indices[:-1], 0] = np.nan
            X_masked[indices[-1], 1] = np.mean(X_masked[indices, 1])
            X_masked[indices[:-1], 1] = np.nan
            X_masked[indices[-1], 2] = np.mean(X_masked[indices, 2])
            X_masked[indices[:-1], 2] = np.nan

            sigma_masked[indices[-1], 0] = np.mean(sigma_masked[indices, 0])
            sigma_masked[indices[:-1], 0] = np.nan
            sigma_masked[indices[-1], 1] = np.mean(sigma_masked[indices, 1])
            sigma_masked[indices[:-1], 1] = np.nan
            sigma_masked[indices[-1], 2] = np.mean(sigma_masked[indices, 2])
            sigma_masked[indices[:-1], 2] = np.nan

            rho_masked[indices[-1]] = np.mean(rho_masked[indices])
            rho_masked[indices[:-1]] = np.nan

            delta_masked[indices[-1]] = np.mean(delta_masked[indices])
            delta_masked[indices[:-1]] = np.nan

            frame_masked[indices[-1]] = np.mean(frame_masked[indices])
            frame_masked[indices[:-1]] = np.nan

print('removed ', len(np.where(np.isnan(X_masked[:,0]))[0]), ' over ', len(X_masked[:,0]))

'''
stdx = np.array(stdx)
stdy = np.array(stdy)
stdz = np.array(stdz)
stdrho = np.array(stdrho)
stddelta = np.array(stddelta)

print(np.mean(stdx))
print(np.mean(stdy))
print(np.mean(stdz))
print(np.mean(stdrho))
print(np.mean(stddelta))
'''

X_treated = np.array([X_masked[~np.isnan(X_masked[:,0]), 0] , X_masked[~np.isnan(X_masked[:,1]), 1] ,X_masked[~np.isnan(X_masked[:,2]), 2]])
sigma_treated = np.array([sigma_masked[~np.isnan(sigma_masked[:,0]), 0] , sigma_masked[~np.isnan(sigma_masked[:,1]), 1] ,sigma_masked[~np.isnan(sigma_masked[:,2]), 2]])
rho_treated = rho_masked[~np.isnan(rho_masked)]
delta_treated = delta_masked[~np.isnan(delta_masked)]
frame_treated = frame_masked[~np.isnan(frame_masked)]

'''

Processing points:   0%|          | 0/139803 [00:00<?, ?it/s]

Processing points: 100%|██████████| 139803/139803 [11:42<00:00, 199.05it/s]

nan
nan
nan
nan
nan



c:\Users\LOCCO\Project_Curie\DSF\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\LOCCO\Project_Curie\DSF\.venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


'\n\nX_treated = np.array([X_masked[~np.isnan(X_masked[:,0]), 0] , X_masked[~np.isnan(X_masked[:,1]), 1] ,X_masked[~np.isnan(X_masked[:,2]), 2]])\nsigma_treated = np.array([sigma_masked[~np.isnan(sigma_masked[:,0]), 0] , sigma_masked[~np.isnan(sigma_masked[:,1]), 1] ,sigma_masked[~np.isnan(sigma_masked[:,2]), 2]])\nrho_treated = rho_masked[~np.isnan(rho_masked)]\ndelta_treated = delta_masked[~np.isnan(delta_masked)]\nframe_treated = frame_masked[~np.isnan(frame_masked)]\n\n'

In [2]:
#Import data treated
data_treated = pd.read_csv(r'C:\Users\LOCCO\Project_Curie\pour_lola\zone1_cc_treated.csv',sep=',')
frame_treated = data_treated['frame'].values
X_treated = data_treated[['x [nm]', 'y [nm]', 'z [nm]']].values
rho_treated = data_treated['rho'].values

In [4]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.colors import hsv_to_rgb

def plot_semicircle_colorbar(ax=None, title='ρ (°)'):
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 2.5), subplot_kw=dict(projection=None))
    
    n_segments = 180
    theta = np.linspace(0, np.pi, n_segments + 1)  # 0 to π (semi-circle)
    
    for i in range(n_segments):
        angle_deg = i  # 0 to 179 degrees
        hue = angle_deg / 180.0
        color = hsv_to_rgb([[hue, 1.0, 1.0]])[0]
        
        # Wedge from theta[i] to theta[i+1]
        wedge = mpatches.Wedge(
            center=(0, 0),
            r=1.0,
            theta1=np.degrees(theta[i]),
            theta2=np.degrees(theta[i+1]),
            width=0.4,        # ring thickness
            color=color
        )
        ax.add_patch(wedge)
    
    # Tick labels at 0°, 45°, 90°, 135°, 180°
    for deg in [0, 45, 90, 135, 180]:
        rad = np.radians(deg)
        x = 1.15 * np.cos(rad)
        y = 1.15 * np.sin(rad)
        ax.text(x, y, f'{deg}°', ha='center', va='center', fontsize=9)
    
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-0.3, 1.4)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=11)
    
    return ax

# --- Use it standalone ---
fig, ax = plt.subplots(figsize=(4, 2.5))
plot_semicircle_colorbar(ax)
plt.tight_layout()
plt.show()

from matplotlib.patches import FancyArrowPatch

def add_scale_bar(ax, length_mum=1, fontsize=9):
    # Get current axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    length = length_mum*1000  # in micrometers, adjust as needed
    # Position: bottom-left corner with some padding
    x_start = xlim[0] + 0.05 * (xlim[1] - xlim[0])
    y_pos   = ylim[0] + 0.04 * (ylim[1] - ylim[0])
    x_end   = x_start + length
    
    # Draw bar
    ax.plot([x_start, x_end], [y_pos, y_pos], 'k-', linewidth=2, solid_capstyle='butt')
    # End ticks
    tick_h = 0.01 * (ylim[1] - ylim[0])
    ax.plot([x_start, x_start], [y_pos - tick_h, y_pos + tick_h], 'k-', linewidth=2)
    ax.plot([x_end,   x_end  ], [y_pos - tick_h, y_pos + tick_h], 'k-', linewidth=2)
    # Label
    ax.text((x_start + x_end) / 2, y_pos + 2 * tick_h,
            f'{length_mum} µm', ha='center', va='bottom', fontsize=fontsize,
            color='black')



In [12]:
fig = plt.figure(figsize=(14, 6))

# Main scatter
ax_scatter = fig.add_axes([0.05, 0.1, 0.7, 0.85])  # [left, bottom, width, height]
norm = matplotlib.colors.Normalize(vmin=0, vmax=180)
sc = ax_scatter.scatter(X_treated[:,0], X_treated[:,1],
                        c=rho_treated, cmap=matplotlib.colormaps['hsv'],
                        norm=norm, s=0.01)
ax_scatter.set_aspect('equal')
ax_scatter.set_title('Localizations colored by ρ')
# After your scatter:
add_scale_bar(ax_scatter, length_mum=1)  # 1 µm bar

# Semi-circle colorbar
ax_cbar = fig.add_axes([0.75, 0.2, 0.22, 0.6])
plot_semicircle_colorbar(ax_cbar, title='ρ (°)')

plt.show()

In [61]:
stdrho

[]

In [6]:
#Petite visualisation 
plt.close('all')
plt.rcParams['figure.figsize'] = [12,12]
hues = rho_treated / 180.0
hsv_colors = np.stack((hues, np.ones_like(hues), np.ones_like(hues)), axis=1)
rgb_colors = hsv_to_rgb(hsv_colors)
plt.scatter(X_treated[0], X_treated[1], c=rgb_colors, s=0.01)
plt.axis('equal')
plt.colorbar(label='Orientation dans le plan (°)')

ValueError: 'c' argument has 138943 elements, which is inconsistent with 'x' and 'y' with size 3.

In [42]:
plt.close('all')
plt.rcParams['figure.figsize'] = [12, 12]

norm = mpl.colors.Normalize(vmin=0, vmax=180)
hsv_cmap = mpl.colormaps['hsv']

sc = plt.scatter(X_treated[0], X_treated[1], 
                 c=rho_treated, cmap=hsv_cmap, norm=norm, s=0.01)
plt.axis('equal')
plt.colorbar(sc, label='Orientation dans le plan (°)')

In [39]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb
import numpy as np

plt.close('all')
fig = plt.figure(figsize=(14, 6))

# --- 3D scatter ---
ax1 = fig.add_subplot(121, projection='3d')
hues = rho_treated / 180.0
hsv_colors = np.stack((hues, np.ones_like(hues), np.ones_like(hues)), axis=1)
rgb_colors = hsv_to_rgb(hsv_colors)

ax1.scatter(X_treated[0], X_treated[1], X_treated[2],
            c=rgb_colors, s=0.01, alpha=0.5)

ax1.set_zlim(X_treated[2].min(), X_treated[2].max())
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax1.set_title('3D localizations')

# --- Color (angle) distribution ---
ax2 = fig.add_subplot(122)
n, bins_rho, patches = ax2.hist(rho_treated, bins=180, range=(0, 180))

# Color each bar by its hue
for patch, left_edge in zip(patches, bins_rho[:-1]):
    hue = left_edge / 180.0
    patch.set_facecolor(hsv_to_rgb([[hue, 1.0, 1.0]])[0])

ax2.set_xlabel('ρ angle (degrees)')
ax2.set_ylabel('Count')
ax2.set_title('Color (angle) distribution')

plt.tight_layout()
plt.show()

In [33]:
#Histogramme 2D de densité

plt.close('all')
plt.figure(figsize=(10, 10))

# Normal histogram
bins = 200
H, xedges, yedges = np.histogram2d(X_treated[0, :], X_treated[1, :], bins=bins)

plt.imshow(H.T, origin='lower', cmap='inferno', extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]], aspect='equal')
plt.colorbar(label='Point density')
plt.title(f'Point density n_bins ={ bins}')
plt.xlabel('X')
plt.ylabel('Y')
plt.show()



In [36]:
import matplotlib as mpl
from matplotlib.widgets import LassoSelector
import sys
from matplotlib.path import Path
sys.path.append(os.path.abspath('..'))
%matplotlib qt
plt.rcParams['figure.figsize'] = [15,15]
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
norm = mpl.colors.Normalize(vmin=20., vmax=160.)
vals = rho_treated / 180.0
sc = ax.scatter(X_treated[0, :], X_treated[1, :], X_treated[2, :],c=rgb_colors, norm=norm, s=0.1)
ax.axis('equal')
cb = plt.colorbar(sc)

plt.rcParams['figure.figsize'] = [15,15]
fig = plt.figure()
ax = fig.add_subplot()
norm = mpl.colors.Normalize(vmin=20., vmax=160.)
vals = rho_treated / 180.0
sc = ax.scatter(X_treated[0, :], X_treated[1, :], c=rgb_colors, norm=norm, s=0.1)
ax.axis('equal')
cb = plt.colorbar(sc)
cb.ax.invert_yaxis()
points = np.column_stack((X_treated[0, :], X_treated[1, :]))
mask2 = np.zeros(len(X_treated[0, :]), dtype=bool) 

def onselect(verts):
    global mask
    path = Path(verts)
    mask = path.contains_points(points) 
    print(mask)

lasso = LassoSelector(ax, onselect)
plt.show()


C:\Users\LOCCO\AppData\Local\Temp\ipykernel_26580\1131666532.py:12: UserWarning: No data for colormapping provided via 'c'. Parameters 'norm' will be ignored
  sc = ax.scatter(X_treated[0, :], X_treated[1, :], X_treated[2, :],c=rgb_colors, norm=norm, s=0.1)
C:\Users\LOCCO\AppData\Local\Temp\ipykernel_26580\1131666532.py:21: UserWarning: No data for colormapping provided via 'c'. Parameters 'norm' will be ignored
  sc = ax.scatter(X_treated[0, :], X_treated[1, :], c=rgb_colors, norm=norm, s=0.1)


In [37]:
import matplotlib.colors as mcolors

# Custom HSV-based colormap (matches your hue encoding)
hsv_cmap = mpl.colormaps['hsv']

norm = mpl.colors.Normalize(vmin=0., vmax=180.)

# 3D plot
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
sc = ax.scatter(X_treated[0, :], X_treated[1, :], X_treated[2, :],
                c=rho_treated, cmap=hsv_cmap, norm=norm, s=0.1)
ax.axis('equal')
cb = plt.colorbar(sc)
cb.set_label('ρ (degrees)')

# 2D plot
fig = plt.figure()
ax = fig.add_subplot()
sc = ax.scatter(X_treated[0, :], X_treated[1, :],
                c=rho_treated, cmap=hsv_cmap, norm=norm, s=0.1)
ax.axis('equal')
cb = plt.colorbar(sc)
cb.set_label('ρ (degrees)')
cb.ax.invert_yaxis()

points = np.column_stack((X_treated[0, :], X_treated[1, :]))
mask2 = np.zeros(len(X_treated[0, :]), dtype=bool)

def onselect(verts):
    global mask2
    path = Path(verts)
    mask2 = path.contains_points(points)
    print(mask2)

lasso = LassoSelector(ax, onselect)
plt.show()

In [ ]:

plt.rcParams['figure.figsize'] = [15,15]
fig = plt.figure()
ax = fig.add_subplot()
norm = mpl.colors.Normalize(vmin=20., vmax=160.)

# ----- Figure layout -----
fig = plt.figure(figsize=(10, 5))

# Polar subplot
ax1 = fig.add_subplot(1, 2, 1, projection='polar')

ax1.bar(bin_centers_full, counts_full, width=width, alpha=0.8)

ax1.set_theta_zero_location("E")   # 0° to the right
ax1.set_theta_direction(1)         # anti-clockwise
ax1.set_yticklabels([])

# PCA
XY = np.column_stack((x[mask], y[mask]))
pca = PCA(n_components=1)
principal_coord = pca.fit_transform(XY).flatten()

# Sort along principal axis
idx = np.argsort(principal_coord)
x_sorted = principal_coord[idx]
z_sorted = z[mask].values[idx]   # use .values if it's pandas

# Moving average window size (adjust!)
window = 150

z_smooth = np.convolve(
    z_sorted,
    np.ones(window)/window,
    mode='valid'
)

# Correct matching x values
x_smooth = x_sorted[:len(z_smooth)]

# Cartesian subplot
ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(x_smooth, z_smooth, color='red', linewidth=2)
ax2.scatter(principal_coord, z[mask], s=5, alpha=0.6)

ax2.set_xlabel("lateral coordinate")
ax2.set_ylabel("z")
ax2.set_aspect('equal')
ax2.set_title("lat–z scatter")
ax2.grid()
ax2.set_ylim(0,800)
plt.tight_layout()
plt.show()
%matplotlib inline
fig = plt.figure(figsize=(5,5))
ax = plt.subplot(111, projection='polar')

# convert to radians
theta = np.deg2rad(eta[mask])

# histogram in [0, π]
counts, bins = np.histogram(theta, bins=100, range=(0, np.pi))

# bin centers
theta_centers = (bins[:-1] + bins[1:]) / 2
width = np.diff(bins)

# duplicate at θ + π
theta_full = np.concatenate([theta_centers, theta_centers + np.pi])
counts_full = np.concatenate([counts, counts])
width_full = np.concatenate([width, width])

# plot
ax.bar(theta_full, counts_full, width=width_full)
# orientation
ax.set_theta_zero_location("N")   # 0° at the top
ax.set_theta_direction(-1)        # clockwise angles

plt.show()
%matplotlib qt
plt.rcParams['figure.figsize'] 
= [15,15]
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
norm = mpl.colors.Normalize(vmin=20., vmax=160.)
vals = eta[mask]
sc = ax.scatter(x[mask] , y[mask], z[mask], c=vals , cmap='coolwarm', norm=norm, s=0.01)
ax.axis('equal')
cb = plt.colorbar(sc)
%matplotlib qt
plt.rcParams['figure.figsize'] = [15,15]
fig = plt.figure()
ax = fig.add_subplot()
norm = mpl.colors.Normalize(vmin=0., vmax=180.)
vals = rho[mask]
sc = ax.scatter(x[mask] , y[mask], c=vals , cmap='hsv', norm=norm, s=0.01)
ax.axis('equal')
cb = plt.colorbar(sc)
cb.ax.invert_yaxis() 

In [ ]:
vals = eta[mask2]
sc = ax.scatter(x[mask2] , z[mask2], c=vals , cmap='coolwarm', norm=norm, s=0.1)
ax.axis('equal')
cb = plt.colorbar(sc)
cb.ax.invert_yaxis()
points = np.column_stack((x, z))
mask = np.zeros(len(x), dtype=bool) 

def onselect(verts):
    global mask
    path = Path(verts)
    mask = path.contains_points(points) & mask2
    print(mask)

lasso = LassoSelector(ax, onselect)
plt.show()
angles = np.deg2rad(rho[mask])
bins = 18
counts, bin_edges = np.histogram(angles, bins=bins, density=True)

bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
width = bin_edges[1] - bin_edges[0]

# Duplicate with π shift
bin_centers_full = np.concatenate([bin_centers, bin_centers + np.pi])
counts_full = np.concatenate([counts, counts])
